In [ ]:
# | default_exp features.wps_panel

In [ ]:
# | export
from __future__ import annotations
import pandas as pd
import numpy as np
import structlog
from kreview.eval_engine import FeatureEvaluator, parse_array

log = structlog.get_logger()

In [ ]:
# | export


class WPSPanelEvaluator(FeatureEvaluator):
    """Extracts WPS nucleosome binding geometries with spectral features.

    For each WPS array, extracts:
    - mean, std (original)
    - peak-to-valley amplitude
    - median absolute deviation
    - spectral max power and dominant frequency (FFT-based periodicity)
    - local_depth scalar (if available)

    Handles both numpy array and string columns from krewlyzer parquets.
    """

    name = "WPSPanel"
    source_file = ".WPS.panel.parquet"
    tier = 2
    category = "epigenetics_and_geometry"

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        extracted = {}
        try:
            if df.empty:
                return extracted
            cols = set(df.columns)

            if "region_type" in cols:
                array_cols = ["wps_nuc", "wps_tf", "prot_frac_nuc", "prot_frac_tf"]
                float_cols = ["local_depth"]
                for row in df.to_dict("records"):
                    rt = (
                        str(row["region_type"])
                        .replace(" ", "_")
                        .replace("-", "_")
                        .replace("|", "_")
                    )

                    for m in float_cols:
                        if m in cols:
                            v = row[m]
                            if v is not None and not (
                                isinstance(v, float) and np.isnan(v)
                            ):
                                extracted[f"{rt}_{m}"] = float(v)

                    for a in array_cols:
                        if a in cols:
                            arr_raw = parse_array(row[a])
                            if arr_raw is not None and len(arr_raw) > 0:
                                arr = np.array(arr_raw)
                                extracted[f"{rt}_{a}_mean"] = float(np.mean(arr))
                                extracted[f"{rt}_{a}_std"] = float(np.std(arr))

                                extracted[f"{rt}_{a}_peak_valley"] = float(
                                    np.max(arr) - np.min(arr)
                                )
                                extracted[f"{rt}_{a}_mad"] = float(
                                    np.median(np.abs(arr - np.median(arr)))
                                )

                                if len(arr) >= 50:
                                    fft_vals = np.abs(np.fft.rfft(arr - arr.mean()))
                                    freqs = np.fft.rfftfreq(len(arr))
                                    if len(fft_vals) > 2:
                                        extracted[f"{rt}_{a}_spectral_max_power"] = (
                                            float(np.max(fft_vals[1:]))
                                        )
                                        extracted[
                                            f"{rt}_{a}_spectral_dominant_freq"
                                        ] = float(freqs[1:][np.argmax(fft_vals[1:])])

            return extracted
        except Exception as e:
            log.exception("extraction_failed", evaluator=self.name, error=str(e))
            return {}

In [ ]:
# | test
from kreview.features.wps_panel import _parse_array

evaluator = WPSPanelEvaluator()
assert len(_parse_array("[0.1 0.2]")) == 2
df = pd.DataFrame(
    [{"region_type": "CTCF", "wps_nuc": "[1.0 2.0 3.0]", "local_depth": 0.5}]
)
res = evaluator.extract(df)
assert res["CTCF_wps_nuc_mean"] == 2.0
assert res["CTCF_local_depth"] == 0.5